In [1]:
import numpy as np
import struct
import sys

In [2]:
######### Reading the images 


def read_images(path):

    with open(path , "rb") as f:
        magic , num_images , rows , cols = struct.unpack(">IIII" , f.read(16))

        data = np.frombuffer(f.read(),dtype=np.uint8)

        images = data.reshape(num_images,rows*cols)

    return images.tolist()


def read_labels(path):

    with open(path , "rb") as f:

        magic , num_labels = struct.unpack(">II" , f.read(8))

        labels = np.frombuffer(f.read(), dtype=np.uint8)

    return labels.tolist()

    


    

In [3]:
########## computing prior 
import math

def compute_priors(labels):

    counts = [0]*10

    for y in labels:
        counts[y] += 1
    
    total = len(labels)
    prior = []

    for i in range(10):
        p = (counts[i] + 1) / (total + 10)
        prior.append(math.log(p))

    
    return prior




In [4]:
def train_discrete(images,labels):
    bins = 32
    counts = [[[0] * bins for _ in range(784)] for _ in range(10)]

    for i in range(len(images)):
        d = labels[i]

        for p in range(784):
            v = images[i][p]
            b = v//8
            counts[d][p][b] +=1


    log_theta = []

    for c in range(10):
        class_prob = []
        
        for p in range(784):
            total_prob = sum(counts[c][p])+32
            pixel_prob = []

            for b in range(32):
                prob = (counts[c][p][b] + 1)/total_prob
                pixel_prob.append(math.log(prob))

            class_prob.append(pixel_prob)

        log_theta.append(class_prob)

    
    return log_theta








In [5]:
def discrete_prediction(image,log_theta,log_priors):

    log_scores = [0]*10

    for c in range(10):
        log_scores[c] = log_priors[c]


    
    for c in range(10):
        for p in range(784):

            v = image[p]
            b = v // 8
            log_scores[c] = log_scores[c] + log_theta[c][p][b]

    

    ### Normalization 

    m = max(log_scores)                 
    exps = [math.exp((s - m) / 50) for s in log_scores]
    Z = sum(exps)                       
    log_posteriors = [e/Z for e in exps]          



    best_class = 0
    best_score = log_posteriors[0]

    for c in range(1,10):
        if log_posteriors[c] > best_score:
            best_class = c
            best_score = log_posteriors[c]


    
    return best_class , log_posteriors

In [6]:
import math
def evaluate_discrete(test_images, test_labels, log_theta, log_priors):
    total = len(test_images)
    errors = 0
    epsilon = 1e-300  # avoid log(0)

    for i in range(total):
        pred, probs = discrete_prediction(test_images[i], log_theta, log_priors)
        true = test_labels[i]

        # Print posterior in log scale
        print("\nPosterior (in log scale):")
        for c in range(10):
            print(f" {c}: {probs[c]:.12f}")

        print(f"Prediction: {pred}, Ans: {true}")

        if pred != true:
            errors += 1

    # Final metrics
    error_rate = errors / total
    accuracy = 1 - error_rate

    print("\nTotal test images:", total)
    print("Errors:", errors)
    print("Error rate:", f"{error_rate:.4f}")
    print("Accuracy:", f"{accuracy:.4f}")

    return errors, total












In [7]:
def train_continuous(images,labels):

    means = [[0.0 for _ in range(784)] for _ in range(10)]
    variances = [[0.0 for _ in range(784)] for _ in range(10)]
    counts = [0] * 10

    for i in range(len(images)):
        c = labels[i]
        counts[c] += 1
        for p in range(784):
            means[c][p] += images[i][p]


    ### mean 
    for c in range(10):
        if counts[c] == 0:
            continue
        for p in range(784):
            means[c][p] /= counts[c]

    ### variance
    for i in range(len(images)):
        c = labels[i]
        for p in range(784):
            diff = images[i][p] - means[c][p]
            variances[c][p] += diff * diff

    for c in range(10):
        if counts[c] == 0:
            continue
        for p in range(784):
            variances[c][p] /= counts[c]
            if variances[c][p] < 1.0:
                variances[c][p] = 1.0 
    


    return means, variances

In [8]:
def continuous_prediction(image,means,variances,log_priors):

    log_scores = [0.0] * 10

    for c in range(10):
        s = log_priors[c]           
        for p in range(784):        
            diff = image[p] - means[c][p]
            var = variances[c][p]
            s += -0.5 * math.log(2 * math.pi * var) - (diff * diff) / (2 * var)
        log_scores[c] = s




    ######### Normalization

    m = max(log_scores)
    exps = [math.exp((s - m) / 50) for s in log_scores]
    Z = sum(exps)
    log_posteriors = [e / Z for e in exps]

    best_class = 0
    best_score = log_posteriors[0]
    for c in range(1, 10):
        if log_posteriors[c] > best_score:
            best_class = c
            best_score = log_posteriors[c]

    return best_class, log_posteriors








In [9]:
def evaluate_continuous(test_images, test_labels, means, variances, log_priors):
    total = len(test_images)
    errors = 0
    epsilon = 1e-300  # avoid log(0)

    for i in range(total):
        pred, probs = continuous_prediction(test_images[i], means, variances, log_priors)
        true = test_labels[i]

        # Print posterior in log scale
        print("\nPosterior (in log scale):")
        for c in range(10):
            print(f" {c}: {probs[c]:.12f}")

        print(f"Prediction: {pred}, Ans: {true}")

        if pred != true:
            errors += 1

    # Final metrics
    error_rate = errors / total
    accuracy = 1 - error_rate

    print("\nTotal test images:", total)
    print("Errors:", errors)
    print("Error rate:", f"{error_rate:.4f}")
    print("Accuracy:", f"{accuracy:.4f}")

    return errors, total



In [10]:
########### Imagination of numbers in Bayesian classifier ###########

def imagination_discrete(log_theta):
    print("\nImagination of numbers in Bayesian classifier:\n")

    bins = 32

    for c in range(10):
        print("Digit:", c)

        for r in range(28):
            row = ""

            for col in range(28):
                p = r*28 + col

                max_bin =0
                max_prob = log_theta[c][p][0]

                for b in range(1,bins):
                    if log_theta[c][p][b] > max_prob:
                        max_prob = log_theta[c][p][b]
                        max_bin = b

                if max_bin * 8 >= 128:
                    row += "1"
                else:
                    row += "0"
            print(row)
        
        print()





def imagination_continuous(means):
    print("\nImagination of numbers in Bayesian classifier:\n")
    
    for c in range(10):
        print("Digit:", c)
        
        for r in range(28):
            row = ""
            
            for col in range(28):
                p = r * 28 + col
                
                if means[c][p] >= 128:
                    row += "1"
                else:
                    row += "0"
            print(row)
        print()


















In [ ]:

def main():
    # Ask for mode safely 
    if len(sys.argv) < 2 or not sys.argv[1].isdigit():
        try:
            mode = int(input("Enter mode (0 = Discrete, 1 = Continuous): "))
        except:
            print("Invalid input! Defaulting to 0.")
            mode = 0
    else:
        mode = int(sys.argv[1])

    # Load MNIST data 
    train_images = read_images(r"train-images.idx3-ubyte_")
    train_labels = read_labels(r"train-labels.idx1-ubyte_")
    test_images  = read_images(r"t10k-images.idx3-ubyte_")
    test_labels  = read_labels(r"t10k-labels.idx1-ubyte_")

    log_priors = compute_priors(train_labels)

    # Mode 0 → Discrete
    if mode == 0:
        output_file = "output_discrete.txt"
        print("\nTraining Discrete Naive Bayes ...")
        log_theta = train_discrete(train_images, train_labels)

        with open(output_file, "w") as f:
            original_stdout = sys.stdout
            sys.stdout = f

            print("Evaluating Discrete Naive Bayes ...\n")
            errors, total = evaluate_discrete(test_images, test_labels, log_theta, log_priors)
            print("\nImagination of numbers in Bayesian classifier:\n")
            imagination_discrete(log_theta)

            # final metrics summary
            error_rate = errors / total
            accuracy = 1 - error_rate
            print("\n================ SUMMARY ================")
            print("Total test images:", total)
            print("Errors:", errors)
            print("Error rate:", f"{error_rate:.4f}")
            print("Accuracy:", f"{accuracy:.4f}")
            print("=========================================\n")

            sys.stdout = original_stdout

        print(f" All output saved to {output_file}")

    # Mode 1 → Continuous
    elif mode == 1:
        output_file = "output_continuous.txt"
        print("\nTraining Continuous Naive Bayes ...")
        means, variances = train_continuous(train_images, train_labels)

        with open(output_file, "w") as f:
            original_stdout = sys.stdout
            sys.stdout = f

            print("Evaluating Continuous Naive Bayes ...\n")
            errors, total = evaluate_continuous(test_images, test_labels, means, variances, log_priors)
            print("\nImagination of numbers in Bayesian classifier:\n")
            imagination_continuous(means)

            # final metrics summary
            error_rate = errors / total
            accuracy = 1 - error_rate
            print("\n================ SUMMARY ================")
            print("Total test images:", total)
            print("Errors:", errors)
            print("Error rate:", f"{error_rate:.4f}")
            print("Accuracy:", f"{accuracy:.4f}")
            print("=========================================\n")

            sys.stdout = original_stdout

        print(f"All output saved to {output_file}")

    else:
        print("Invalid mode! Use 0 for Discrete or 1 for Continuous.")


if __name__ == "__main__":
    main()



Training Continuous Naive Bayes ...
All output saved to output_continuous.txt
